# n=100: make the numbers readable

Four notebooks of proxy experiments converged on differences **smaller than the measurement error**.
The bottleneck is n, not ideas.

## The arithmetic

The overlap statistic is hypergeometric — k=10 draws from ~74 candidates with 10 successes:

```
per-example SD  ~ 10.1 points
n = 18   ->  SE 2.4 pts  ->  95% band +/- 4.7 pts
n = 90   ->  SE 1.1 pts  ->  95% band +/- 2.1 pts
```

At n=18 almost every gap we cared about sat inside the band, and `CTRL random` once scored 18.3%
against a 13.5% floor and looked "significant". At n≈90 the current gaps become decidable.

## What this run does differently

* **`N_PER_TYPE = 10`** -> ~100 examples instead of 20.
* **Everything in one pass** — LOO ground truth, both attention labels, both gradient teachers, and
  the question-free predictors — so no later notebook has to re-derive anything.
* **Paired tests.** Every previous table compared arms *against chance only*. This one reports
  per-arm 95% CIs plus **paired** Wilcoxon tests between arms, which is the comparison we have
  actually been trying to make all along.
* **Checkpoints every 10 examples**, so a disconnect costs minutes rather than the whole run.

Cost: ~86 forward-equivalents per example. The 20-example run took 0.9 min for the LOO part, so
budget roughly **10–20 min** for 100 including the attention and gradient passes.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words matplotlib scipy
!rm -rf /content/text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git /content/text_vision_attention_map
%cd /content/text_vision_attention_map

In [ ]:
import importlib.util, os, sys, math, glob, json, time, gc
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.stats import norm, wilcoxon

sys.path.insert(0, os.getcwd())

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import rater_selection as RS
import visual_selection as VS

DATA_ROOT   = "/content/drive/MyDrive/wearvqa_gaze_only"
OUT         = "/content/drive/MyDrive/wearvqa_n100.pt"
SINKF       = "/content/drive/MyDrive/sink_mask_smolvlm2_n12.pt"
MODEL_ID    = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
NULL_PROMPT = "Describe the image."
N_PER_TYPE  = 10          # <- the change: 2 -> 10
GROUP       = 1           # keep 1 for comparability with the n=20 run
LOSS_SCALE  = 1e3
N_SINK_PROBE = 12

model, processor, device = S._load_smolvlm(MODEL_ID)
tokenizer = processor.tokenizer

types = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
samples = []
for t in types:
    for jp in sorted(glob.glob(os.path.join(DATA_ROOT, t, "*.json")))[:N_PER_TYPE]:
        m = json.load(open(jp)); ip = jp[:-5] + ".jpg"
        if os.path.exists(ip) and "gaze" in m and m.get("response"):
            samples.append(dict(type=t, img_path=ip, question=m["question"],
                                answer=m["response"], gaze=m["gaze"]))
print(f"{len(types)} types | {len(samples)} samples")
for t in types:
    print(f"   {t:<38} {sum(s['type']==t for s in samples)}")

## 2. Machinery

In [ ]:
def build_inputs(image, question, answer=""):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    full = processor(text=prompt + answer, images=[image], return_tensors="pt").to(device)
    only = processor(text=prompt, images=[image], return_tensors="pt")
    return full, int(only["input_ids"].shape[1])

@torch.no_grad()
def answer_logprob(inp, n_prompt, attention_mask=None):
    kw = dict(inp)
    if attention_mask is not None:
        kw["attention_mask"] = attention_mask
    logits = model(**kw).logits[0].float()
    lp = torch.log_softmax(logits[:-1], dim=-1)
    return float(lp.gather(-1, inp["input_ids"][0, 1:, None]).squeeze(-1)[n_prompt - 1:].sum())

def patch_groups(L, g):
    G_ = int(round(math.sqrt(L)))
    if g <= 1:
        return [[i] for i in range(L)]
    return [[r*G_+c for r in range(r0, min(r0+g, G_)) for c in range(c0, min(c0+g, G_))]
            for r0 in range(0, G_, g) for c0 in range(0, G_, g)]

def grad_capturing_eager():
    try:
        from transformers.models.llama.modeling_llama import repeat_kv
    except Exception:
        def repeat_kv(h, n):
            b, kv, s, d = h.shape
            return h if n == 1 else h[:, :, None].expand(b, kv, n, s, d).reshape(b, kv*n, s, d)
    def fn(module, query, key, value, attention_mask=None, scaling=None, dropout=0.0, **kw):
        if scaling is None:
            scaling = getattr(module, "scaling", 1.0)
        ng = getattr(module, "num_key_value_groups", 1)
        k, v = repeat_kv(key, ng), repeat_kv(value, ng)
        attn = torch.matmul(query, k.transpose(2, 3)) * scaling
        if attention_mask is not None:
            attn = attn + attention_mask[:, :, :, : k.shape[-2]]
        probs = F.softmax(attn, dim=-1, dtype=torch.float32).to(query.dtype)
        if probs.requires_grad:
            probs.retain_grad()
        module._probs = probs
        out = torch.matmul(F.dropout(probs, p=dropout, training=module.training), v)
        return out.transpose(1, 2).contiguous(), None
    return fn

def find_decoder_layers(model, n=24):
    hits = [(nm, m) for nm, m in model.named_modules()
            if isinstance(m, torch.nn.ModuleList) and len(m) == n]
    for nm, m in hits:
        if any(t in nm for t in ("text", "language", "llm")):
            return m
    return hits[0][1]

dec = find_decoder_layers(model)

def masks_for(ids, n_prompt):
    iid = S._find_image_token_id(model, processor)
    pad = tokenizer.pad_token_id
    im  = ids == iid
    tm  = (ids != iid) & (ids != (pad if pad is not None else -10**9))
    return im, tm, torch.nonzero(im).squeeze(-1), torch.nonzero(tm).squeeze(-1)

print("ok")

## 3. Sink mask from 12 examples

In [ ]:
if os.path.exists(SINKF):
    blob_ = torch.load(SINKF, weights_only=False)
    sinks, sink_score = blob_["sink_mask"].bool(), blob_["sink_score"]
    print(f"loaded sink mask: {torch.nonzero(sinks).squeeze(-1).tolist()}")
else:
    step = max(1, len(samples) // N_SINK_PROBE)
    sc = []
    for s in samples[::step][:N_SINK_PROBE]:
        o = S.make_smolvlm_output(image=S.load_image(s["img_path"]), question=s["question"])
        if o is None:
            continue
        sc.append(VS.sink_scores(o.post_softmax, o.image_token_mask, o.text_token_mask,
                                 is_post_softmax=True))
        del o; gc.collect(); torch.cuda.empty_cache()
    sink_score = VS.aggregate_sink_scores(sc)
    sinks = VS.detect_sinks(sink_score)
    L_v0 = sink_score.numel()
    torch.save({"model": MODEL_ID, "L_v": L_v0, "sink_mask": sinks,
                "sink_score": sink_score, "n_probe": len(sc)}, SINKF)
    print(VS.sink_report(sink_score, sinks))

## 4. Main pass — LOO + attention + gradients + question-free features

In [ ]:
def process(s):
    img = S.load_image(s["img_path"])
    inp, n_prompt = build_inputs(img, s["question"], s["answer"])
    ids = inp["input_ids"][0].cpu()
    im, tm, img_cols, tpos = masks_for(ids, n_prompt)
    L_v = len(img_cols)
    cand = VS.candidate_mask(L_v, exclude=[sinks[:L_v]])

    # ---- LOO ground truth ----
    base = answer_logprob(inp, n_prompt)
    drops = torch.zeros(L_v)
    for grp in patch_groups(L_v, GROUP):
        am = inp["attention_mask"].clone(); am[0, img_cols[grp]] = 0
        drops[grp] = (base - answer_logprob(inp, n_prompt, am)) / len(grp)

    # ---- one forward+backward: attention labels AND both gradient teachers ----
    store = {}
    def pre_hook(_m, args):
        h = args[0]
        if h.requires_grad:
            h.retain_grad(); store["h"] = h
        return None
    hh = dec[0].register_forward_pre_hook(pre_hook)
    patched = S._patch_eager_globals(grad_capturing_eager())
    try:
        model.zero_grad(set_to_none=True)
        out = model(**inp)
        lp = torch.log_softmax(out.logits[0].float()[:-1], dim=-1)
        logp = lp.gather(-1, inp["input_ids"][0, 1:, None]).squeeze(-1)[n_prompt-1:].sum()
        (logp * LOSS_SCALE).backward()
    finally:
        S._unpatch_eager_globals(patched); hh.remove()

    ans_rows = torch.tensor([int(p) for p in tpos.tolist() if int(p) >= n_prompt])
    rec = dict(**s, base_logp=base, drops=drops)

    if "h" in store and store["h"].grad is not None:
        h = store["h"][0].detach().float(); gr = store["h"].grad[0].detach().float()
        rec["grad_x_input"] = (gr[img_cols] * h[img_cols]).sum(-1).abs().cpu()
        E = h[img_cols].cpu()
        En = F.normalize(E, dim=-1)
        gp = min(int(math.sqrt(L_v))-1, int(s["gaze"]["y_norm"]*int(math.sqrt(L_v)))) * int(math.sqrt(L_v)) \
             + min(int(math.sqrt(L_v))-1, int(s["gaze"]["x_norm"]*int(math.sqrt(L_v))))
        rec["gp"] = gp
        rec["gaze_sim"] = (En @ En[gp])
        rec["distinct"] = 1 - (En @ F.normalize(E.mean(0), dim=-1))
        rec["tok_norm"] = E.norm(dim=-1)

    post, ag = {}, torch.zeros(L_v)
    for m in model.modules():
        p = getattr(m, "_probs", None)
        if p is None:
            continue
        if p.shape[-1] == len(ids) and p.shape[-2] == len(ids):
            post[int(getattr(m, "layer_idx", len(post)))] = p[0].detach().float().cpu()
            if p.grad is not None and len(ans_rows):
                blk = (p[0].detach() * p.grad[0].detach()).abs().float()
                ag += blk[:, ans_rows][:, :, img_cols].sum(1).sum(0).cpu()
        del m._probs
    rec["attn_x_grad"] = ag

    if post:
        maps, tp, _ = RS.sliced_maps_from_full(
            {l: torch.log(a.clamp_min(1e-12)) for l, a in post.items()}, im, tm)
        tt  = tokenizer.convert_ids_to_tokens(ids[tp].tolist())
        isa = torch.tensor([int(p) >= n_prompt for p in tp.tolist()])
        am_ = RS.content_text_mask(tt, tokenizer) & isa
        if int(am_.sum()) == 0:
            am_ = isa
        rec["imp_a"], *_ = VS.image_importance(maps, am_, cand_mask=cand)
        rq = RS.select_important_text_tokens(maps, text_tokens=tt, tokenizer=tokenizer,
                                             question=s["question"], pct=0.5).rater_mask
        rec["imp_q"], *_ = VS.image_importance(maps, rq, cand_mask=cand)

    # ---- null-prompt pass: question-free attention ----
    ninp, _ = build_inputs(img, NULL_PROMPT)
    nids = ninp["input_ids"][0].cpu()
    nim, ntm, _, ntp = masks_for(nids, 0)
    patched = S._patch_eager_globals(S._make_raw_capturing_eager(None))
    try:
        with torch.no_grad():
            model(**ninp)
    finally:
        S._unpatch_eager_globals(patched)
    nraw = {}
    for m in model.modules():
        r = getattr(m, "_raw_attn_scores", None)
        if r is not None and r.shape[-1] == len(nids) and r.shape[-2] == len(nids):
            nraw[int(getattr(m, "layer_idx", len(nraw)))] = r[0].float()
        for a in ("_raw_attn_scores", "_post_attn"):
            if hasattr(m, a):
                delattr(m, a)
    if nraw:
        nmaps, ntpos, _ = RS.sliced_maps_from_full(nraw, nim, ntm)
        ntt = tokenizer.convert_ids_to_tokens(nids[ntpos].tolist())
        ncm = RS.content_text_mask(ntt, tokenizer)
        if int(ncm.sum()) == 0:
            ncm = torch.ones(len(ntt), dtype=torch.bool)
        rec["null_imp"], *_ = VS.image_importance(nmaps, ncm, cand_mask=cand)

    del out, post, store, nraw
    gc.collect(); torch.cuda.empty_cache()
    return rec


data, t0 = [], time.time()
if os.path.exists(OUT):
    data = torch.load(OUT, weights_only=False)
    print(f"resuming from {len(data)} cached examples")

for i in range(len(data), len(samples)):
    try:
        data.append(process(samples[i]))
    except Exception as e:
        print(f"  [skip {i} {samples[i]['type']}] {type(e).__name__}: {e}")
        gc.collect(); torch.cuda.empty_cache()
        continue
    if i == 0:
        per = time.time() - t0
        print(f"first example {per:.1f}s -> ETA {per*len(samples)/60:.1f} min")
    if (i + 1) % 10 == 0:
        torch.save(data, OUT)
        print(f"  {i+1}/{len(samples)}  ({(time.time()-t0)/60:.1f} min)  [checkpoint]")

torch.save(data, OUT)
L_v = data[0]["drops"].numel(); G = int(round(math.sqrt(L_v))); N = len(data)
print(f"\ndone in {(time.time()-t0)/60:.1f} min | {N} examples | L_v={L_v} -> {OUT}")

## 5. Scoring — with confidence intervals and **paired** tests

Every earlier table compared arms against chance only. These report per-arm 95% CIs and paired
Wilcoxon tests, which is the comparison we have been trying to make since the first bake-off.

In [ ]:
DX_SIG, DY_SIG, DX_MU, DY_MU = 2.69, 1.72, 0.95, -0.20

def blob(gp, sc=DX_SIG, sr=DY_SIG, oc=DX_MU, orr=DY_MU):
    r0, c0 = divmod(gp, G); r0, c0 = r0 + orr, c0 + oc
    return torch.tensor([-(((i//G - r0)/sr)**2 + ((i%G - c0)/sc)**2) for i in range(L_v)])

def isotropic(gp):
    r0, c0 = divmod(gp, G)
    return torch.tensor([-math.hypot(i//G - r0, i % G - c0) for i in range(L_v)])

def fovea_mask(gp):
    r0, c0 = divmod(gp, G)
    m = torch.zeros(L_v, dtype=torch.bool)
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            r, c = r0+dr, c0+dc
            if 0 <= r < G and 0 <= c < G:
                m[r*G+c] = True
    return m

g = torch.Generator().manual_seed(0)

def predictors(d):
    gp = d["gp"]
    p = {"attn x grad  [Q]":        d.get("attn_x_grad"),
         "grad x input [Q]":        d.get("grad_x_input"),
         "attention imp_a [Q]":     d.get("imp_a"),
         "attention imp_q [Q]":     d.get("imp_q"),
         "token norm (image)":      d.get("tok_norm"),
         "null-prompt attn (image)":d.get("null_imp"),
         "distinctiveness (image)": d.get("distinct"),
         "gaze-sim = untrained FRM":d.get("gaze_sim"),
         "gaze blob (geometry)":    blob(gp),
         "isotropic gaze prox":     isotropic(gp),
         "center (nothing)":        isotropic(L_v // 2),
         "CTRL random":             torch.rand(L_v, generator=g)}
    return {k: v for k, v in p.items() if v is not None}

def score(mask_fn, title, k=10):
    rows, chances, used = defaultdict(list), [], 0
    for d in data:
        if "gp" not in d:
            continue
        mask = mask_fn(d)
        cidx = torch.nonzero(mask, as_tuple=False).squeeze(-1)
        if float(torch.topk(d["drops"][cidx], k).values[-1]) <= 1e-6:
            continue
        used += 1; chances.append(k / int(mask.sum()))
        gt = set(cidx[torch.topk(d["drops"][cidx], k).indices].tolist())
        for n_, v in predictors(d).items():
            rows[n_].append(len(set(cidx[torch.topk(v[cidx], k).indices].tolist()) & gt) / k)
    chance = float(np.mean(chances))
    order = sorted(rows, key=lambda n_: -np.mean(rows[n_]))
    top = order[0]
    print(f"\n=== {title}")
    print(f"    precision@{k}   chance {chance:.1%}   n={used}   "
          f"95% band +/- {1.96*np.std(rows[top],ddof=1)/math.sqrt(used)*100:.1f} pts")
    print(f"    {'predictor':<26}{'prec':>7}{'95% CI':>16}{'vs rand':>9}{'vs top':>9}")
    print("    " + "-" * 68)
    rnd = np.array(rows["CTRL random"])
    for n_ in order:
        a = np.array(rows[n_]); se = a.std(ddof=1)/math.sqrt(len(a))
        lo, hi = a.mean()-1.96*se, a.mean()+1.96*se
        def pw(b):
            try:
                return f"{wilcoxon(a, b)[1]:.3f}" if not np.allclose(a, b) else "  --"
            except Exception:
                return "   n/a"
        print(f"    {n_:<26}{a.mean():>7.1%}  [{lo:>5.1%},{hi:>6.1%}]"
              f"{pw(rnd):>9}{pw(np.array(rows[top])):>9}")
    return rows

r1 = score(lambda d: VS.candidate_mask(L_v, exclude=[sinks[:L_v]]),
           "ALL CANDIDATES (sinks removed)")

In [ ]:
r2 = score(lambda d: VS.candidate_mask(L_v, exclude=[sinks[:L_v], fovea_mask(d["gp"])]),
           "FOVEA EXCLUDED - context you are NOT looking at  [FRM's actual job]")

## 6. Verdict

The whole point of n≈100 is that the **`vs top` column** is now interpretable. Read it, not the
raw ordering.

* **`CTRL random` back at chance** with a tight CI -> the referee is behaving; at n=18 it once drifted
  to 18.3% and looked significant.
* **Any arm whose CI excludes `CTRL random`'s** is a real effect. Anything overlapping it is not,
  no matter where it sits in the ordering.
* **`gaze-sim` (untrained FRM) vs `gaze blob`** in the fovea-excluded table is the FRM question:
  does the gaze token's *embedding* beat its *position*? At n=18 it did not (17.8% vs 21.1%, both ns).
* **`token norm` still on top** would confirm the LOO referee retains a high-norm artifact component
  even after sink removal — i.e. the ground truth is still partly measuring "deleting a high-norm
  token destabilises the model". That is a finding about the referee, not about the labels.
* **Question-conditioned arms `[Q]` well above the question-free ones** measures exactly how much of
  the signal FRM is structurally unable to reach.

Whatever this says, it is the last proxy experiment worth running. If the question-free ceiling is
meaningfully above the geometry floor, the next step is to generate LOO labels at scale and **train
the module with real `W_q, W_k`** — identity projections underestimate a trained FRM, and no further
proxy will settle that.